## Data validation

In [45]:
import pandas as pd

df = pd.read_csv("merged_dataset_2000.csv")

print("Shape:", df.shape)

print("\nMissing values")
print(df.isnull().sum())

print("\nLeak label distribution")
print(df["leak_label"].value_counts())

print("\nLeak types")
print(df["leak_type"].value_counts())

print("\nPreview")
print(df.head())

Shape: (98000, 21)

Missing values
scenario             0
time_index           0
leak_label           0
leak_type            0
leak_score           0
mean_demand          0
max_demand           0
min_demand           0
std_demand           0
demand_range         0
mean_pressure        0
max_pressure         0
min_pressure         0
std_pressure         0
pressure_range       0
mean_flow            0
max_flow             0
min_flow             0
std_flow             0
flow_range           0
flow_demand_ratio    0
dtype: int64

Leak label distribution
45000    2000
86400    2000
46800    2000
48600    2000
50400    2000
52200    2000
54000    2000
55800    2000
57600    2000
59400    2000
61200    2000
63000    2000
64800    2000
66600    2000
68400    2000
70200    2000
72000    2000
73800    2000
75600    2000
77400    2000
79200    2000
81000    2000
82800    2000
1800     2000
43200    2000
41400    2000
19800    2000
3600     2000
5400     2000
7200     2000
9000     2000
10800    2

time_index = row index of your simulation timestep.Just a count step with no unit

## Correct unrealistic values

In [46]:
duplicates = df.duplicated().sum()
print("Duplicates:", duplicates)

df = df.drop_duplicates()

Duplicates: 0


In [47]:
# df = df.sort_values(['scenario', 'time_index']).reset_index(drop=True)

In [48]:
# print(df[['scenario','time_index']].head(20))


In [49]:
# Clip demand, flow, pressure at the 1st and 99th percentiles per column
cols_to_clip = ['mean_demand', 'max_demand', 'min_demand', 'std_demand',
                'mean_pressure', 'max_pressure', 'min_pressure', 'std_pressure',
                'mean_flow', 'max_flow', 'min_flow', 'std_flow', 'flow_demand_ratio']

for col in cols_to_clip:
    lower = df[col].quantile(0.01)
    upper = df[col].quantile(0.99)
    df[col] = df[col].clip(lower=lower, upper=upper)

print(" Clipped to remove outliers.")

 Clipped to remove outliers.


In [50]:
# Show first 10 rows after clipping
print(df.head(10))

        scenario  time_index  leak_label   leak_type  leak_score  \
0  scenario_0001           0           0  continuous    0.110162   
1  scenario_0001           1        1800  continuous    0.110162   
2  scenario_0001           2        3600  continuous    0.110162   
3  scenario_0001           3        5400  continuous    0.110162   
4  scenario_0001           4        7200  continuous    0.110162   
5  scenario_0001           5        9000  continuous    0.110162   
6  scenario_0001           6       10800  continuous    0.110162   
7  scenario_0001           7       12600  continuous    0.110162   
8  scenario_0001           8       14400  continuous    0.110162   
9  scenario_0001           9       16200  continuous    0.110162   

    mean_demand    max_demand  min_demand   std_demand  demand_range  ...  \
0  5.878788e-10      0.104167   -1.538585     0.277663      1.642752  ...   
1  5.454545e+01   1800.000000   -1.538585   313.339904   1801.538585  ...   
2  1.090909e+02   36

sample = df[df['scenario'] == 'scenario_0052']
print(sample['time_index'].head(20))

df.groupby('scenario')['time_index'].apply(lambda x: x.is_monotonic_increasing).value_counts()

In [51]:
df.to_csv("cleaned_merged_dataset.csv", index=False)
print("Cleaned dataset saved successfully!")


Cleaned dataset saved successfully!


In [52]:
df1 = pd.read_csv("cleaned_merged_dataset.csv")

In [53]:
# Sort for time-series correctness
df1 = df1.sort_values(by=["scenario", "time_index"]).reset_index(drop=True)

print("Loaded dataset:", df1.shape)

Loaded dataset: (98000, 21)


# Add Targets

In [54]:
#TARGETS
# Instant leak (binary)
df1["instant_leak"] = df1["leak_label"].astype(int)

# Slow leak (3-day window example)
WINDOW = 72   # adjust if needed

df1["slow_leak"] = (
    df1.groupby("scenario")["instant_leak"]
    .transform(lambda x: x.rolling(WINDOW, min_periods=1).max())
)
print("Targets created.")

Targets created.


# Feature Engineering

In [57]:
# 7. PREPARE FEATURES
# ------------------------------
df1 = df1.loc[:, ~df1.columns.duplicated()]

exclude = ["scenario", "time_index", "leak_label", "instant_leak", "slow_leak"]
sensor_cols = [col for col in df1.columns if col not in exclude]

# Convert safely to numeric
df1[sensor_cols] = df1[sensor_cols].apply(pd.to_numeric, errors='coerce').fillna(0)

print("Sensor columns:", len(sensor_cols))

# ------------------------------
# 8. FEATURE ENGINEERING
# ------------------------------
print("Starting feature engineering...")

features_dict = {}
total_sensors = len(sensor_cols)

for idx, col in enumerate(sensor_cols, start=1):

    # Show progress
    if idx % 5 == 0 or idx == 1:
        print(f"Processing {idx}/{total_sensors}: {col}")

    lag = df1.groupby("scenario")[col].shift(1)

    features_dict[f"{col}_lag1"] = lag
    features_dict[f"{col}_diff"] = df1[col] - lag

    features_dict[f"{col}_roll_mean"] = df1.groupby("scenario")[col].transform(
        lambda x: x.rolling(window=6, min_periods=1).mean()
    )

    features_dict[f"{col}_roll_std"] = df1.groupby("scenario")[col].transform(
        lambda x: x.rolling(window=6, min_periods=1).std()
    )

# Combine features
features_df = pd.DataFrame(features_dict, index=df1.index)

df1 = pd.concat([df1, features_df], axis=1)

# Final cleanup
df1 = df1.fillna(0)

print("Feature engineering completed!")
print("Final shape:", df1.shape)


Sensor columns: 90
Starting feature engineering...
Processing 1/90: leak_type
Processing 5/90: min_demand
Processing 10/90: min_pressure
Processing 15/90: min_flow
Processing 20/90: leak_type_diff
Processing 25/90: leak_score_roll_mean
Processing 30/90: mean_demand_roll_std
Processing 35/90: min_demand_lag1
Processing 40/90: std_demand_diff
Processing 45/90: demand_range_roll_mean
Processing 50/90: mean_pressure_roll_std
Processing 55/90: min_pressure_lag1
Processing 60/90: std_pressure_diff
Processing 65/90: pressure_range_roll_mean
Processing 70/90: mean_flow_roll_std
Processing 75/90: min_flow_lag1
Processing 80/90: std_flow_diff
Processing 85/90: flow_range_roll_mean
Processing 90/90: flow_demand_ratio_roll_std
Feature engineering completed!
Final shape: (98000, 455)
